# Layer 0 Linear Probing Skyline

Run linear probing on `layer_0` representations using refactored probing modules, then build a skyline table.


In [ ]:
from pathlib import Path
import os
import pandas as pd

from prob.paths_and_io import get_project_root, get_exp_dirs, load_X_from_pt, load_y_by_ids
from prob.prob_config import get_ds_load_config
from prob.prob_registry import LINEAR_PROBES
from prob.prob_run import run_cv_search


In [ ]:
# Ensure stable root discovery for this notebook session.
project_root = get_project_root()
os.environ["HOME_PROJ_DIR"] = str(project_root)
print("Project root:", project_root)

# --- experiment config (edit as needed) ---
TARGET_FILE = "weighted_hb_score.pt"
LAYER_NUM = 0
RANDOM_STATE = 96
N_SPLITS_CV = 5
TEST_SIZE = 0.1
N_JOBS = -1

# Reuse tuned params for similar reruns.
REUSE_BEST_PARAMS = True
BEST_PARAMS_CACHE_DIR = project_root / "data" / "probing" / "_shared_best_params"


In [ ]:
prob_config = get_ds_load_config(
    target_file=TARGET_FILE,
)

X = load_X_from_pt(prob_config.output_dir, layer_num=LAYER_NUM)
y = load_y_by_ids(
    prob_config.output_dir,
    target_dir=prob_config.target_dir,
    targets_file=TARGET_FILE,
)

target_name = Path(TARGET_FILE).stem
print("X shape:", X.shape)
print("y shape:", y.shape)
print("target:", target_name)


In [ ]:
runs = []

for entry in LINEAR_PROBES:
    name = entry["name"]
    estimator = entry["estimator"]
    param_grid = entry.get("param_grid", {})

    exp_dirs = get_exp_dirs(
        prob_config.output_dir,
        target=target_name,
        prob_model=name,
        layer_num=LAYER_NUM,
    )

    search, metrics, _ = run_cv_search(
        X,
        y,
        estimator,
        param_grid,
        n_splits=N_SPLITS_CV,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
        exp_dirs=exp_dirs,
        model_name=name,
        run_stats=True,
        reuse_best_params=REUSE_BEST_PARAMS,
        best_params_cache_dir=BEST_PARAMS_CACHE_DIR,
    )

    runs.append({
        "model": name,
        "layer": LAYER_NUM,
        **metrics,
        **search.best_params_,
    })

results_df = pd.DataFrame(runs)
results_df


In [ ]:
# Skyline: higher R2 is better, lower RMSE is better.
skyline_df = (
    results_df
    .sort_values(["r2", "rmse"], ascending=[False, True])
    .reset_index(drop=True)
)

display(skyline_df[["model", "r2", "rmse", "mae", "fit_seconds"]])
best_model = skyline_df.iloc[0]
print(f"Best skyline model: {best_model['model']} | R2={best_model['r2']:.4f} | RMSE={best_model['rmse']:.4f}")


In [ ]:
# Save notebook-level skyline summary.
skyline_out = Path(prob_config.output_dir) / target_name / "experiments" / f"layer_{LAYER_NUM}_linear_skyline.csv"
skyline_out.parent.mkdir(parents=True, exist_ok=True)
skyline_df.to_csv(skyline_out, index=False)
print("Saved skyline:", skyline_out)
